In [6]:
import json, pickle, openai
import numpy as np
import os
import httpx

# 设置代理（确保在 Jupyter Notebook 中也能使用代理）
os.environ['HTTP_PROXY'] = 'http://10.21.194.77:1082'
os.environ['HTTPS_PROXY'] = 'http://10.21.194.77:1082'
os.environ['NO_PROXY'] = 'localhost,127.0.0.1,10.21.194.77'

## Reference:
# https://platform.openai.com/docs/models 
# https://github.com/openai/openai-python/blob/main/README.md
# https://zenodo.org/records/10030426 (GenePT embeddings)
# https://zenodo.org/records/10833191 (GenePT embeddings)

# API 密钥
api_key="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"
client = openai.OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)

# with open(f"./input_data/GenePT_embedding/NCBI_cleaned_summary_of_genes.json", 'r') as file:
#     NCBI_summary_of_genes = json.load(file)
    
A1BG_summary_text = "Official Symbol A1BGprovided by HGNC Official Full Name alpha-1-B glycoproteinprovided by HGNC Primary source HGNC:HGNC:5 See related Ensembl:ENSG00000121410 MIM:138670; AllianceGenome:HGNC:5 Gene type protein coding RefSeq status REVIEWED Organism Homo sapiens Lineage Eukaryota; Metazoa; Chordata; Craniata; Vertebrata; Euteleostomi; Mammalia; Eutheria; Euarchontoglires; Primates; Haplorrhini; Catarrhini; Hominidae; Homo Also known as A1B; ABG; GAB; HYST2477 Summary The protein encoded by this gene is a plasma glycoprotein of unknown function. The protein shows sequence similarity to the variable regions of some immunoglobulin supergene family member proteins. [provided by RefSeq, Jul 2008] Orthologs mouse all"
print('A1BG_summary_text:', A1BG_summary_text)
embedding = client.embeddings.create(input = A1BG_summary_text, model="openai/text-embedding-ada-002", encoding_format="float").data[0].embedding
print('A1BG_summary_text:', A1BG_summary_text)
print('A1BG_embedding:', embedding)

# NCBI_summary_to_embedding = {}
# for key, text in sorted(NCBI_summary_of_genes.items()):
#     if key not in NCBI_summary_to_embedding:
#         if NCBI_summary_of_genes[key] == '': 
#             NCBI_summary_to_embedding[key] = np.zeros(1536) # it's hard coded to be 0
#         else:
#             embedding = client.embeddings.create(input = [text], model="text-embedding-ada-002").data[0].embedding
#             NCBI_summary_to_embedding[key] = np.array(embedding)
            
# # with open(f"./input_data/GPT_3_5_gene_embeddings.pickle", "wb") as fp:
# #     pickle.dump(NCBI_summary_to_embedding, fp)

A1BG_summary_text: Official Symbol A1BGprovided by HGNC Official Full Name alpha-1-B glycoproteinprovided by HGNC Primary source HGNC:HGNC:5 See related Ensembl:ENSG00000121410 MIM:138670; AllianceGenome:HGNC:5 Gene type protein coding RefSeq status REVIEWED Organism Homo sapiens Lineage Eukaryota; Metazoa; Chordata; Craniata; Vertebrata; Euteleostomi; Mammalia; Eutheria; Euarchontoglires; Primates; Haplorrhini; Catarrhini; Hominidae; Homo Also known as A1B; ABG; GAB; HYST2477 Summary The protein encoded by this gene is a plasma glycoprotein of unknown function. The protein shows sequence similarity to the variable regions of some immunoglobulin supergene family member proteins. [provided by RefSeq, Jul 2008] Orthologs mouse all
A1BG_summary_text: Official Symbol A1BGprovided by HGNC Official Full Name alpha-1-B glycoproteinprovided by HGNC Primary source HGNC:HGNC:5 See related Ensembl:ENSG00000121410 MIM:138670; AllianceGenome:HGNC:5 Gene type protein coding RefSeq status REVIEWED Or

In [5]:
#### We provide an example to get GTP to complete content for gene ALPP. For your use case it could be any other genes of interest.
gene_name_to_GPT_response = {}
gene_list = ['ALPP']
for gene in sorted(gene_list):
    if gene not in gene_name_to_GPT_response:
        completion = client.chat.completions.create(model="gpt-3.5-turbo", messages=[{"role": "user", "content": f"Tell me about gene {gene}"}]) 
        # role: system is the instruction, user is the input, assistant is the output. input to model: role + content for each message.
        gene_name_to_GPT_response[gene] = completion.choices[0].message.content

# Let's also insepct the results to make sure that GPT-3.5 was not hallucinating here --- indeed the output from this API call looks biologically relevant:
gene_name_to_GPT_response['ALPP']

'The gene ALPP, also known as alkaline phosphatase, placental type, is a gene responsible for encoding placental alkaline phosphatase, an enzyme that plays a key role in normal placental development during pregnancy. This gene is located on chromosome 2 in humans and is expressed primarily in the placenta, although it can also be found in other tissues such as the liver, bone, and kidney.\n\nAlkaline phosphatase is a critical enzyme in the body that helps to regulate various cellular processes, including mineralization of bones, and is essential for fetal growth and development. Mutations or abnormalities in the ALPP gene can lead to conditions such as hypophosphatasia, a rare genetic disorder characterized by defective bone mineralization and other skeletal abnormalities.\n\nResearch on the ALPP gene continues to uncover its importance in pregnancy and development, as well as its potential role in diseases and disorders affecting the skeletal system. Studies on this gene may lead to n

Output from GPT: `'Gene ALPP, also known as Alkaline Phosphatase, Placental-Like (ALPPL2), is a gene that encodes an enzyme called Alkaline phosphatase placental-like. This gene is located on human chromosome 2q37.3.\n\nAlkaline phosphatase is an enzyme that removes phosphate groups from various molecules, and it is involved in many important physiological processes. ALPP is primarily expressed in the placenta, but it can also be found in other tissues like liver, kidney, and intestine.\n\nThe ALPP gene is thought to play a role in placental development and function, as well as in fetal and postnatal bone metabolism. Studies have shown that mutations or dysregulation of ALPP expression can lead to various disorders and diseases. For example, decreased ALPP activity has been associated with hypophosphatasia, a rare genetic disorder characterized by defective bone mineralization.\n\nIn addition, ALPP expression has been linked to certain types of cancer. Increased ALPP activity has been observed in some malignancies, such as colorectal cancer and hepatocellular carcinoma, and it has been suggested as a potential marker for these cancers.\n\nOverall, the ALPP gene and its encoded enzyme, Alkaline phosphatase placental-like, have important roles in placental development, bone metabolism, and potentially in certain cancers. Further research is still needed to fully understand the exact mechanisms and implications of this gene in various physiological processes and diseases.'`